# Fine-tuning Qwen3-8B on the forensic science dataset (Google Colab)

QLoRA fine-tuning with [Unsloth](https://github.com/unslothai/unsloth) on a free T4 GPU.
Everything you need to keep lives on Google Drive, so a disconnected session loses nothing
and you can continue training the same model later.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then upload your `*.jsonl`
files (chatml format, the generator's default) to `MyDrive/forensics-llm/data/`.

```
MyDrive/forensics-llm/
├── data/              ← your *.jsonl datasets
├── checkpoints/<run>/ ← trainer checkpoints: resume after a disconnect
├── adapters/<run>/    ← the trained LoRA weights + run_info.json: continue training from here
└── gguf/<run>/        ← (optional) quantised model for Ollama / llama.cpp
```

The LoRA adapter (~170 MB) **is** your fine-tuned model: the new weights on top of the
base model. Keep it — it is what you load to train further. GGUF is only for running the
model locally.

## 1. Install

In [ ]:
%pip install -q unsloth

## 2. Settings

First run: leave `CONTINUE_FROM = None`.
Later runs: set `CONTINUE_FROM` to a previous `RUN_NAME` and give the new run a new name.

In [ ]:
RUN_NAME = "v1"  # this run's name: adapters/<RUN_NAME>, checkpoints/<RUN_NAME>
CONTINUE_FROM = None  # e.g. "v1" to keep training a saved adapter

BASE_MODEL = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048  # the longest generated examples are ~1,100 tokens
EPOCHS = 2
LEARNING_RATE = 1e-4 if CONTINUE_FROM else 2e-4
LORA_RANK = 16
EVAL_SHARE = 0.05  # held-out examples for the validation loss
SEED = 3407

## 3. Google Drive

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/forensics-llm")
DATA = ROOT / "data"
CHECKPOINTS = ROOT / "checkpoints" / RUN_NAME
ADAPTER = ROOT / "adapters" / RUN_NAME
for path in (DATA, CHECKPOINTS, ADAPTER.parent):
    path.mkdir(parents=True, exist_ok=True)

files = sorted(DATA.glob("*.jsonl"))
assert files, f"Upload your *.jsonl files to {DATA}"
for file in files:
    print(file.name)
if ADAPTER.exists() and not CONTINUE_FROM and not any(CHECKPOINTS.glob("checkpoint-*")):
    print(f"Warning: {ADAPTER} already exists and will be overwritten; change RUN_NAME?")

## 4. Model

In [ ]:
from unsloth import FastLanguageModel

if CONTINUE_FROM:
    # Loads the base model and the saved adapter on top of it.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(ROOT / "adapters" / CONTINUE_FROM),
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    for name, param in model.named_parameters():
        if "lora_" in name:
            param.requires_grad_(True)
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_RANK,
        lora_dropout=0,
        bias="none",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
        ],
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
model.print_trainable_parameters()

## 5. Data

Every line is `{"messages": [...]}`. The Qwen3 chat template keeps the `<think>` block of
reasoning examples and adds an empty one to the rest, as in Qwen3's non-thinking mode.

In [ ]:
import hashlib
import json

from datasets import Dataset

records = []
for file in files:
    for number, line in enumerate(file.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        messages = json.loads(line).get("messages")
        assert messages, f"{file.name}:{number}: not chatml; generate with -f chatml"
        records.append({"text": tokenizer.apply_chat_template(messages, tokenize=False)})

split = Dataset.from_list(records).train_test_split(test_size=EVAL_SHARE, seed=SEED)
train_data, eval_data = split["train"], split["test"]
print(f"{len(train_data)} training, {len(eval_data)} validation examples")
print(train_data[0]["text"][:1500])

## 6. Trainer

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=SFTConfig(
        output_dir=str(CHECKPOINTS),
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=2,
        seed=SEED,
        report_to="none",
    ),
)
# Learn only the assistant's answers, not the system prompt and the questions.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

## 7. Train

If Colab disconnects, run cells 1–6 again with the same settings and then this one: it
picks up from the last checkpoint on Drive.

In [ ]:
resume = any(CHECKPOINTS.glob("checkpoint-*"))
print("Resuming from the last checkpoint" if resume else "Starting from scratch")
stats = trainer.train(resume_from_checkpoint=resume or None)
print(stats)

## 8. Save the new weights

The adapter and `run_info.json` (base model, data files, settings, losses and the history
of previous runs) go to `adapters/<RUN_NAME>` on Drive.

In [ ]:
from datetime import UTC, datetime

model.save_pretrained(str(ADAPTER))
tokenizer.save_pretrained(str(ADAPTER))

history = []
if CONTINUE_FROM:
    previous = ROOT / "adapters" / CONTINUE_FROM / "run_info.json"
    if previous.exists():
        info = json.loads(previous.read_text())
        history = [*info.pop("history", []), info]

eval_losses = [log["eval_loss"] for log in trainer.state.log_history if "eval_loss" in log]
run_info = {
    "run": RUN_NAME,
    "date": datetime.now(UTC).isoformat(timespec="seconds"),
    "base_model": BASE_MODEL,
    "continued_from": CONTINUE_FROM,
    "data": [
        {"file": f.name, "sha256": hashlib.sha256(f.read_bytes()).hexdigest()} for f in files
    ],
    "examples": {"train": len(train_data), "eval": len(eval_data)},
    "settings": {
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lora_rank": LORA_RANK,
        "max_seq_length": MAX_SEQ_LENGTH,
    },
    "train_loss": stats.training_loss,
    "eval_loss": eval_losses[-1] if eval_losses else None,
    "history": history,
}
(ADAPTER / "run_info.json").write_text(json.dumps(run_info, ensure_ascii=False, indent=2))
print(f"Saved to {ADAPTER}:", sorted(p.name for p in ADAPTER.iterdir()))

## 9. Try it

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

SYSTEM = (
    "You are an AI assistant for forensic science and criminalistics. You help "
    "investigators, forensic experts, crime scene specialists and law students: you "
    "accurately explain concepts, methods and tools of forensic technology, investigative "
    "tactics and investigation methodology, based on scientific evidence and the law."
)


def ask(question, thinking=False):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=thinking
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.6 if thinking else 0.7,
        top_p=0.95 if thinking else 0.8,
        top_k=20,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )


ask("What distinguishes an organized from a disorganized crime scene?")

## 10. (Optional) Export to GGUF for Ollama

Takes 15–30 minutes. The model is built on the Colab disk and only the ~5 GB `.gguf` file and
a `Modelfile` are copied to Drive (a free Drive has 15 GB, a full 16-bit model is 16 GB).
Locally: download `gguf/<RUN_NAME>/` and run `ollama create forensics -f Modelfile`.

In [ ]:
import re
import shutil

GGUF = ROOT / "gguf" / RUN_NAME
GGUF.mkdir(parents=True, exist_ok=True)
model.save_pretrained_gguf("/content/gguf", tokenizer, quantization_method="q4_k_m")

exported = [f for f in Path("/content").glob("gguf*/*.gguf") if "q4_k_m" in f.name.lower()]
assert exported, "No .gguf file was produced; see the log above"
shutil.copy(exported[0], GGUF / exported[0].name)
print("Copied", exported[0].name)

# Unsloth writes a Modelfile with the Qwen3 chat template; point it at the copied file.
generated = next(Path("/content").glob("gguf*/Modelfile"), None)
modelfile = re.sub(r"^FROM .*$", "", generated.read_text() if generated else "", flags=re.M)
modelfile = f"FROM ./{exported[0].name}\n{modelfile.strip()}\n"
if "SYSTEM" not in modelfile:
    system = SYSTEM.replace('"', '\\"')
    modelfile += f'SYSTEM "{system}"\n'
(GGUF / "Modelfile").write_text(modelfile)
print(f"Saved to {GGUF}")

## Continuing training later

1. Put the new `*.jsonl` files in `data/`. Leave roughly 20–30% of the old data there too:
   training only on new material makes the model forget the old.
2. In the settings, set `CONTINUE_FROM = "v1"` and `RUN_NAME = "v2"`. `BASE_MODEL` must stay
   the same, because an adapter only works with the base model it was trained on.
3. *Runtime → Run all.* The new adapter goes to `adapters/v2`. `v1` stays untouched, so you can
   always go back to it.